<div dir="rtl" align="right">

# رگرسیون Lasso (منظم‌سازی L1)

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
Lasso هم ضرایب بزرگ را جریمه می‌کند، ولی با **قدر مطلق** به جای مربع:

$$J(\mathbf w)=\frac{1}{2n}\sum_i(y_i-\hat y_i)^2+\alpha\sum_j\vert w_j\vert$$

این تفاوت کوچک نتیجه بزرگی دارد: Lasso بسیاری از ضرایب را **دقیقاً صفر** می‌کند. یعنی خودش ویژگی‌های بی‌فایده را کنار می‌گذارد؛ **انتخاب ویژگی خودکار**.

## چرا صفر می‌شود؟ (تصویر هندسی)
جواب جایی است که منحنی‌های تراز خطا (بیضی‌های نارنجی) به ناحیه مجاز ضرایب برخورد می‌کنند. ناحیه L2 دایره است و برخورد معمولاً جایی است که هر دو ضریب غیرصفرند. ناحیه L1 لوزی است و **گوشه‌هایش روی محورها** هستند؛ برخورد اغلب در گوشه رخ می‌دهد، یعنی یکی از ضرایب صفر است.

> [چپ: Ridge (دایره). راست: Lasso (لوزی) که در گوشه، w1 را صفر می‌کند.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/l1_l2.png)

*چپ: Ridge (دایره). راست: Lasso (لوزی) که در گوشه، w1 را صفر می‌کند.*

## چطور حل می‌شود؟ (کاهش مختصاتی)
قدر مطلق در صفر مشتق ندارد، پس فرمول بسته وجود ندارد. scikit-learn هر بار یک ضریب را بهینه می‌کند و از **آستانه نرم** استفاده می‌کند:

$$w_j\leftarrow S(\rho_j,\alpha)=\text{sign}(\rho_j)\max(\vert\rho_j\vert-\alpha,0)$$

که $\rho_j$ همبستگی ویژگی $j$ با باقی‌مانده فعلی است. اگر $\vert\rho_j\vert\le\alpha$ باشد، ضریب دقیقاً صفر می‌شود.

| $\rho_j$ | $\alpha$ | ضریب جدید |
|---|---|---|
| 3.0 | 1.0 | 2.0 |
| 0.6 | 1.0 | 0 |
| −2.5 | 1.0 | −1.5 |

## مسیر ضرایب
با زیاد شدن $\alpha$ ضرایب **یکی‌یکی** صفر می‌شوند. آخرین ضرایبی که زنده می‌مانند، مهم‌ترین ویژگی‌ها هستند:

> [مسیر Lasso روی داده دیابت.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/lasso_path.png)

*مسیر Lasso روی داده دیابت.*

[[lab:reg_lasso]]

در آزمایشگاه، $\alpha$ را زیاد کن و تعداد «ضرایب غیرصفر» را دنبال کن. میله‌های خاکستری ضرایبی هستند که صفر شده‌اند.



</div>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

d = load_diabetes(as_frame=True)
X = StandardScaler().fit_transform(d.data); y = d.target
table = {}
for a in [0.01, 1, 5, 20, 50]:
    table[f"alpha={a}"] = Lasso(alpha=a).fit(X, y).coef_
coef = pd.DataFrame(table, index=d.data.columns).round(1)
print(coef)
print("Non-zero per alpha:", (coef != 0).sum().to_dict())

<div dir="rtl" align="right">



## Lasso یا Ridge؟
| موقعیت | انتخاب بهتر |
|---|---|
| فکر می‌کنی فقط چند ویژگی واقعاً مهم‌اند | Lasso |
| همه ویژگی‌ها کمی اثر دارند | Ridge |
| ویژگی‌های خیلی همبسته داری | Ridge یا Elastic Net (Lasso یکی را تصادفی برمی‌دارد) |
| ویژگی‌ها از نمونه‌ها بیشترند | Lasso حداکثر n ویژگی انتخاب می‌کند؛ Elastic Net بهتر است |



</div>

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = load_diabetes(as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(d.data, d.target, test_size=0.25, random_state=0)
m = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0)).fit(X_tr, y_tr)
print("Chosen alpha:", round(m[-1].alpha_, 3))
print("Kept features:", list(d.data.columns[m[-1].coef_ != 0]))
print("Test R2:", round(m.score(X_te, y_te), 3))

<div dir="rtl" align="right">

### تمرین

تابع آستانه نرم soft(rho, alpha) را بنویس و جدول درس را با آن بازتولید کن.

</div>

In [ ]:
# Your answer here

<div dir="rtl" align="right">

### تمرین

۵۰ ویژگی تصادفی بی‌فایده به داده دیابت اضافه کن. Lasso چند تا از آن‌ها را نگه می‌دارد؟

</div>

In [ ]:
# Your answer here